[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# A Complete Data Layer &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell writes the exports as the notebook's Setup did, writes the project its worked
examples wrote, builds `college.db` with the migration, and loads it. Run it first. The tasks follow
one another, as changes to a project do, and the last cell removes the scratch folder.


In [1]:
import csv
import os
import re
import shlex
import shutil
import subprocess
import sys
from datetime import date
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("alembic")
except PackageNotFoundError:                                        # Colab has no Alembic: install the version this notebook runs
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore", "alembic==1.20.0"],
                   check=True)

import sqlalchemy
from sqlalchemy import create_engine, event, func, select
from sqlalchemy.exc import IntegrityError
from sqlalchemy.orm import Session
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

DATA = SCRATCH / "data"
DATA.mkdir()
CODES = [code for code, title, department, credits in COURSES]
TERM_NAMES = [name for name, starts_on in TERMS]
EMAILS = [email for name, email, program, started_on in STUDENTS]
ENROLLMENT_ROWS = [(EMAILS[student - 1], CODES[(section - 1) % 10], TERM_NAMES[(section - 1) // 10], status, grade or "")
                   for student, section, status, grade in ENROLLMENTS]
EXPORTED = {
    "courses.csv": (["code", "title", "department", "credits"], COURSES),
    "terms.csv": (["name", "starts_on"], TERMS),
    "sections.csv": (["course", "term", "capacity"],
                     [(CODES[course - 1], TERM_NAMES[term - 1], capacity) for course, term, capacity in SECTIONS]),
    "students.csv": (["name", "email", "program", "started_on"], STUDENTS),
    "enrollments.csv": (["email", "course", "term", "status", "grade"], ENROLLMENT_ROWS + [
        ("znakamura@college.edu", "STA-200", "Spring 2026", "enrolled", ""),   # nobody with this email
        ENROLLMENT_ROWS[0],                                                     # the first row, sent twice
        ("areyes@college.edu", "PSY-101", "Spring 2026", "dropped", ""),        # a status the table refuses
    ]),
}
for name, (header, rows) in EXPORTED.items():
    with open(DATA / name, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(rows)
print({name: len(rows) for name, (header, rows) in EXPORTED.items()})


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine


os.environ["NO_COLOR"] = "1"                                        # no terminal codes in what the commands print
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"                         # no compiled copy of a file rewritten within a second
sys.dont_write_bytecode = True                                      # and none from this notebook's own imports
engine = college_engine(DATABASE)                                   # connects to nothing until it is used


def alembic(*arguments):
    """Run an alembic command in the project folder, and print what it printed, less the lines every command repeats."""
    engine.dispose()                                                # the notebook's own connections close first
    done = subprocess.run(
        [sys.executable, "-m", "alembic", *arguments], cwd=SCRATCH, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    lines = done.stdout.replace(f"{SCRATCH.resolve()}{os.sep}", "").splitlines()
    if "Traceback (most recent call last):" in lines:              # the error's own line, not Python's files
        error = [line for line in lines if re.match(r"[\w.]+(Error|Exception): ", line)][-1]
        lines = lines[:lines.index("Traceback (most recent call last):")] + ["Traceback (most recent call last): ...", error]
    print("$", shlex.join(["alembic", *arguments]))
    for line in lines:
        if not any(noise in line for noise in ("Context impl", "Will assume", "setting up autogenerate plugin")):
            print("   ", line)


def edit(path, old, new):
    """Replace the one place in a file where old appears with new."""
    source = Path(path).read_text()
    assert source.count(old) == 1, f"{old!r} appears {source.count(old)} times in {path}"
    Path(path).write_text(source.replace(old, new))


def pytest_report(*arguments, folder=SCRATCH):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    report = re.sub(r"0x[0-9a-f]+", "0x...", report)                    # the memory addresses of objects
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


print("alembic", version("alembic"), "| pytest", version("pytest"))

PROJECT_FILES = {
    "models.py": r'''"""The college's tables, as classes."""
from datetime import date

from sqlalchemy import CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"
''',
    "database.py": r'''"""The one place the project makes an engine: SQLite with foreign keys enforced."""
from sqlalchemy import create_engine, event


def make_engine(url):
    engine = create_engine(url, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True                          # the PRAGMA does nothing inside a transaction
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False

    return engine
''',
    "load.py": r'''"""Load the old system's CSV exports into the college's database."""
import csv
from datetime import date

from sqlalchemy import select
from sqlalchemy.exc import IntegrityError

from models import Course, Enrollment, Section, Student, Term


def read_rows(path):
    """Every row of a CSV file, as a dictionary keyed by the header, which may begin with a byte-order mark."""
    with open(path, newline="", encoding="utf-8-sig") as file:
        return list(csv.DictReader(file))


def load_reference(session, folder):
    """Courses, terms, sections and students: everything an enrollment refers to."""
    for row in read_rows(folder / "courses.csv"):
        session.add(Course(code=row["code"], title=row["title"], department=row["department"], credits=int(row["credits"])))
    for row in read_rows(folder / "terms.csv"):
        session.add(Term(name=row["name"], starts_on=date.fromisoformat(row["starts_on"])))
    session.flush()
    courses = {course.code: course for course in session.scalars(select(Course))}
    terms = {term.name: term for term in session.scalars(select(Term))}
    for row in read_rows(folder / "sections.csv"):
        session.add(Section(course=courses[row["course"]], term=terms[row["term"]], capacity=int(row["capacity"])))
    for row in read_rows(folder / "students.csv"):
        session.add(Student(name=row["name"], email=row["email"], program=row["program"],
                            started_on=date.fromisoformat(row["started_on"])))
    session.commit()


def load_enrollments(session, path):
    """Load every enrollment the database accepts, each in a savepoint of its own, and return the rest's line and reason."""
    students = dict(session.execute(select(Student.email, Student.id)).all())
    sections = {(code, term): section_id for section_id, code, term in
                session.execute(select(Section.id, Course.code, Term.name).join(Section.course).join(Section.term))}
    refused = []
    for line, row in enumerate(read_rows(path), start=2):           # line 1 is the header
        try:
            with session.begin_nested():
                session.add(Enrollment(student_id=students[row["email"]], section_id=sections[row["course"], row["term"]],
                                       status=row["status"], grade=row["grade"] or None))
        except KeyError as error:
            refused.append((line, f"unknown: {error.args[0]}"))
        except IntegrityError as error:
            refused.append((line, str(error.orig)))
    session.commit()
    return refused
''',
    "queries.py": r'''"""The registrar's questions, each answered by the database in one statement."""
from sqlalchemy import case, func, select

from models import Course, Enrollment, Section, Student, Term

POINTS = case(                                                      # grade points in tenths, so that every sum is whole
    {"A": 40, "A-": 37, "B+": 33, "B": 30, "B-": 27, "C+": 23, "C": 20, "C-": 17, "D": 10, "F": 0},
    value=Enrollment.grade,
)


def transcript(session, email):
    """A student's courses in the order they were taken: term, course code, grade."""
    return session.execute(
        select(Term.name, Course.code, Enrollment.grade)
        .join(Enrollment.student).join(Enrollment.section).join(Section.term).join(Section.course)
        .where(Student.email == email)
        .order_by(Term.starts_on, Course.code)
    ).all()


def standing(session, term):
    """For every program: the students with grades in the term, those on the dean's list, and those below 2.0."""
    per_student = (
        select(Enrollment.student_id,
               func.sum(POINTS * Course.credits).label("quality"),
               func.sum(Course.credits).label("credits"))
        .join(Enrollment.section).join(Section.course).join(Section.term)
        .where(Term.name == term, Enrollment.grade.is_not(None))
        .group_by(Enrollment.student_id)
        .subquery()
    )
    report = (
        select(Student.program, func.count(),
               func.sum(case((per_student.c.quality >= 30 * per_student.c.credits, 1), else_=0)),
               func.sum(case((per_student.c.quality < 20 * per_student.c.credits, 1), else_=0)))
        .join(per_student, per_student.c.student_id == Student.id)
        .group_by(Student.program)
        .order_by(Student.program)
    )
    return session.execute(report).all()
''',
    "conftest.py": r'''from pathlib import Path

import pytest
from alembic import command
from alembic.config import Config
from sqlalchemy.orm import Session

from database import make_engine
from load import load_reference

HERE = Path(__file__).parent


@pytest.fixture(scope="session")
def migrations(tmp_path_factory):
    """The project's Alembic settings, pointed at a new database of the run's own."""
    config = Config(HERE / "alembic.ini")
    config.set_main_option("sqlalchemy.url", f"sqlite:///{tmp_path_factory.mktemp('db') / 'test.db'}")
    return config


@pytest.fixture(scope="session")
def engine(migrations):
    """The test database, built by the migrations, with the exports' courses, terms, sections and students."""
    command.upgrade(migrations, "head")
    engine = make_engine(migrations.get_main_option("sqlalchemy.url"))
    with Session(engine) as session:
        load_reference(session, HERE / "data")
    yield engine
    engine.dispose()


@pytest.fixture
def session(engine):
    """A session for one test, inside a transaction that is rolled back when the test ends."""
    with engine.connect() as connection:
        transaction = connection.begin()
        with Session(bind=connection, join_transaction_mode="create_savepoint") as session:
            yield session
        transaction.rollback()
''',
    "test_data_layer.py": r'''from pathlib import Path

from alembic import command

from load import load_enrollments
from queries import standing, transcript

DATA = Path(__file__).parent / "data"


def test_the_migrations_match_the_models(migrations, engine):
    command.check(migrations)                                       # raises if autogenerate would find a change


def test_three_rows_are_refused(session):
    refused = load_enrollments(session, DATA / "enrollments.csv")
    assert [line for line, reason in refused] == [230, 231, 232]


def test_every_student_had_grades_in_fall_2025(session):
    load_enrollments(session, DATA / "enrollments.csv")
    students = sum(row[1] for row in standing(session, "Fall 2025"))
    assert students == 25


def test_a_transcript_runs_in_term_order(session):
    load_enrollments(session, DATA / "enrollments.csv")
    terms = [term for term, code, grade in transcript(session, "cmartin@college.edu")]
    assert terms == ["Fall 2025"] * 3 + ["Spring 2026"] * 3
''',
    "build.py": r'''"""Build the college's database from the exports, from nothing: python build.py DATABASE"""
import sys
from pathlib import Path

from alembic import command
from alembic.config import Config
from sqlalchemy import func, select
from sqlalchemy.orm import Session

from database import make_engine
from load import load_enrollments, load_reference
from models import Enrollment, Student
from queries import standing

HERE = Path(__file__).parent
url = f"sqlite:///{HERE / sys.argv[1]}"

config = Config(HERE / "alembic.ini")
config.set_main_option("sqlalchemy.url", url)
command.upgrade(config, "head")

engine = make_engine(url)
with Session(engine) as session:
    load_reference(session, HERE / "data")
    refused = load_enrollments(session, HERE / "data" / "enrollments.csv")
    print(session.scalar(select(func.count()).select_from(Student)), "students,",
          session.scalar(select(func.count()).select_from(Enrollment)), "enrollments loaded")
    for line, reason in refused:
        print(f"refused, line {line}: {reason}")
    for program, students, deans, below in standing(session, "Fall 2025"):
        print(f"{program:<17} {students} students, {deans} on the dean's list, {below} below 2.0")
engine.dispose()
''',
}
for name, text in PROJECT_FILES.items():
    (SCRATCH / name).write_text(text)

alembic("init", "migrations")
ini = SCRATCH / "alembic.ini"
ini.write_text(re.sub(r"^sqlalchemy\.url = .*$", "sqlalchemy.url = sqlite:///college.db", ini.read_text(), flags=re.M))
edit(SCRATCH / "migrations" / "env.py", "target_metadata = None",
     "from models import Base\n\ntarget_metadata = Base.metadata")
alembic("revision", "--autogenerate", "-m", "the college", "--rev-id", "0001")
alembic("upgrade", "head")

sys.path.insert(0, str(SCRATCH))
from load import load_enrollments, load_reference
from models import Enrollment
from queries import standing, transcript

with Session(engine) as session:
    load_reference(session, DATA)
    print(len(load_enrollments(session, DATA / "enrollments.csv")), "rows refused")


{'courses.csv': 10, 'terms.csv': 4, 'sections.csv': 40, 'students.csv': 25, 'enrollments.csv': 231}
alembic 1.20.0 | pytest 8.4.2
$ alembic init migrations
    Creating directory migrations ...  done
    Creating directory migrations/versions ...  done
    Generating migrations/script.py.mako ...  done
    Generating migrations/env.py ...  done
    Generating migrations/README ...  done
    Generating alembic.ini ...  done
    Please edit configuration/connection/logging settings in alembic.ini before proceeding.
$ alembic revision --autogenerate -m 'the college' --rev-id 0001
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'courses'
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'students'
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'terms'
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'sections'
    INFO  [alembic.autogenerate.compare.tables] Detected added table 'enrollments'
    Generating mi

**1.** Aoife O'Brien's transcript.


In [2]:
with Session(engine) as session:
    for term, code, grade in transcript(session, "aobrien@college.edu"):
        print(f"{term:<12} {code:<8} {grade or 'in progress'}")


Fall 2024    CHE-110  F
Fall 2024    CSC-201  B
Fall 2024    PSY-101  C
Spring 2025  ENG-105  D
Spring 2025  MAT-120  B-
Spring 2025  STA-200  A-
Fall 2025    BIO-101  C
Fall 2025    HIS-110  B
Fall 2025    MAT-121  F
Spring 2026  CHE-110  in progress
Spring 2026  CSC-101  in progress
Spring 2026  PSY-101  in progress


Twelve courses: Aoife O'Brien started in Fall 2024 and has taken three every term since.


**2.** Two new enrollments, one for a section that does not exist.


In [3]:
EXTRA = SCRATCH / "extra.csv"
with open(EXTRA, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["email", "course", "term", "status", "grade"])
    writer.writerow(["areyes@college.edu", "BIO-101", "Fall 2025", "completed", "B"])
    writer.writerow(["areyes@college.edu", "ART-100", "Fall 2025", "completed", "A"])

with Session(engine) as session:
    print(load_enrollments(session, EXTRA))
    print(session.scalar(select(func.count()).select_from(Enrollment)), "enrollments")


[(3, "unknown: ('ART-100', 'Fall 2025')")]
229 enrollments


The row for BIO-101 went in, and the college has no ART-100, so the lookup refused the other before
the database was asked, with the course and the term it could not find.


**3.** A new query, the fullest sections of a term.


In [4]:
import importlib

import queries

(SCRATCH / "queries.py").write_text((SCRATCH / "queries.py").read_text() + """

def seats_taken(session, term):
    # Every section of a term with its number of enrollments, the fullest first.
    return session.execute(
        select(Course.code, func.count(Enrollment.student_id).label("taken"))
        .join(Section.course).join(Section.term).outerjoin(Section.enrollments)
        .where(Term.name == term)
        .group_by(Section.id, Course.code)
        .order_by(func.count(Enrollment.student_id).desc(), Course.code)
    ).all()
""")
importlib.reload(queries)

with Session(engine) as session:
    print(queries.seats_taken(session, "Spring 2026")[:3])


[('BIO-101', 8), ('CHE-110', 8), ('CSC-101', 8)]


`importlib.reload` runs the changed file again, since this notebook imported `queries` before the
function was there. The outer join keeps a section nobody has joined, with 0.


**4.** The loader refuses a grade the college does not give.


In [5]:
edit(SCRATCH / "load.py", "from models import Course, Enrollment, Section, Student, Term\n",
     "from models import Course, Enrollment, Section, Student, Term\n\n"
     "GRADES = {\"A\", \"A-\", \"B+\", \"B\", \"B-\", \"C+\", \"C\", \"C-\", \"D\", \"F\"}\n")
edit(SCRATCH / "load.py", "    for line, row in enumerate(read_rows(path), start=2):           # line 1 is the header\n",
     "    for line, row in enumerate(read_rows(path), start=2):           # line 1 is the header\n"
     "        if row[\"grade\"] and row[\"grade\"] not in GRADES:\n"
     "            refused.append((line, f\"unknown grade: {row['grade']}\"))\n"
     "            continue\n")

(SCRATCH / "test_grades.py").write_text("""
from load import load_enrollments


def test_an_unknown_grade_is_refused(session, tmp_path):
    export = tmp_path / "enrollments.csv"
    export.write_text("email,course,term,status,grade\\nareyes@college.edu,BIO-101,Fall 2025,completed,Q\\n")
    refused = load_enrollments(session, export)
    assert refused == [(2, "unknown grade: Q")]
""")
print(pytest_report("-q", "test_grades.py"))


.                                                                        [100%]
1 passed


The check goes in the loader because the table has none, and `tmp_path` gives the test a folder of
its own for the one-row export.


**5.** An index added to the models, and the test that notices.


In [6]:
edit(SCRATCH / "models.py", "    program: Mapped[str] = mapped_column(String(50))",
     "    program: Mapped[str] = mapped_column(String(50), index=True)")
lines = pytest_report("-q", "test_data_layer.py").splitlines()
print("\n".join([line for line in lines if line.startswith(("E ", "FAILED"))] + lines[-1:]))

alembic("revision", "--autogenerate", "-m", "index students by program", "--rev-id", "0002")
print(pytest_report("-q", "test_data_layer.py"))


E           alembic.util.exc.AutogenerateDiffsDetected: New upgrade operations detected: [('add_index', Index('ix_students_program', Column('program', String(length=50), table=<students>, nullable=False)))]
FAILED test_data_layer.py::test_the_migrations_match_the_models - alembic.uti...
1 failed, 3 passed
$ alembic revision --autogenerate -m 'index students by program' --rev-id 0002
    INFO  [alembic.autogenerate.compare.constraints] Detected added index 'ix_students_program' on '('program',)'
    Generating migrations/versions/0002_index_students_by_program.py ...  done
....                                                                     [100%]
4 passed


Only the first test failed. An index changes nothing a load or a query needs, so every other test
passed, and `command.check` is the one that noticed a model the migrations do not describe, and said
which operation was missing. With revision 0002 in `versions`, the test database is built with the
index, and the check has nothing left to report.


**6.** A second database from nothing.


In [7]:
done = subprocess.run([sys.executable, "build.py", "second.db"], cwd=SCRATCH, capture_output=True, text=True)
print(done.stdout.splitlines()[0])

for name in ("second.db", "college.db"):
    with Session(college_engine(SCRATCH / name)) as session:
        print(f"{name:<11}", session.scalar(select(func.count()).select_from(Enrollment)), "enrollments")


25 students, 228 enrollments loaded
second.db   228 enrollments
college.db  229 enrollments


The new database has the 228 enrollments of the exports. `college.db` has one more, the enrollment
task 2 loaded, which is in no export: rebuilding from the exports would lose it, which is why a real
migration from an old system ends with the old system switched off.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [A Complete Data Layer](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/20-a-complete-data-layer.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
